# DA3METRIC-LARGE — Complete Visual Trend Analysis

**Pipeline overview**

| Step | Description |
|------|-------------|
| 1 | Index KITTI Eigen-Split (test + optional train) |
| 2 | Load DA3METRIC-LARGE and run raw (unscaled) depth inference |
| 3 | Mask predicted depth with valid LiDAR pixels; clip at 80 m / 40 m / no-clip |
| 4 | Compute difference maps `LiDAR − Pred` (three variants) |
| 5 | Overlay difference maps on the original RGB image (custom diverging colormap) |
| 6 | Save overlay PNGs + colorbars; write per-frame metrics + point-count CSV |
| 7 | Load pre-generated binary masks; compute per-mask and per-category metrics |
| 8 | Explanatory analysis: histograms, bar charts, scatter plots, heatmaps |

> **No scaling is applied at any stage.** Only raw metric depth from DA3 is used.

## 0 · GPU Selection & Imports

In [1]:
# ── Optional: pick a specific GPU ─────────────────────────────────────────────
# import os; os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import os, sys, glob, json, math, logging, warnings, re
from pathlib import Path
from copy import deepcopy
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
import torchvision.transforms as T

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

# ── DA3 ────────────────────────────────────────────────────────────────────────
from depth_anything_3.api import DepthAnything3

print("Imports OK  |  CUDA:", torch.cuda.is_available())

[WARN ] Dependency `gsplat` is required for rendering 3DGS. Install via: pip install git+https://github.com/nerfstudio-project/gsplat.git@0b4dddf04cb687367602c01196913cde6a743d70
Imports OK  |  CUDA: True


## 1 · Configuration  *(edit paths and thresholds here)*

In [2]:
DEVICE    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CACHE_DIR = "/STORAGE/Arka/cache"

# ── Dataset ───────────────────────────────────────────────────────────────────
KITTI_ROOT = "/STORAGE/Arka/Downloads/KITTI/kitti_eigen_split_dataset_test"
TEST_ROOT  = os.path.join(KITTI_ROOT, "test")
TRAIN_ROOT = os.path.join(KITTI_ROOT, "train")
USE_TRAIN  = False          # ← set True to also process train split

# KITTI LiDAR GT layout (proj_depth/groundtruth)
# Depth PNG path pattern (relative to KITTI_ROOT):
#   <split>/<date>/<drive>/proj_depth/groundtruth/image_02/<frame>.png

# ── Pre-generated binary masks (from inst_seg_notebook_final.ipynb) ───────────
MASK_ROOT     = "/STORAGE/Arka/a_MDE_Code/1_Anaysis/Output"
MIN_MASK_SCORE = 0.40

# ── Depth clipping variants ────────────────────────────────────────────────────
# Each entry: (label, max_depth_metres or None)
CLIP_VARIANTS = [
    ("clip80",  80.0),
    ("clip40",  40.0),
    ("noclip",  None),
]
MIN_DEPTH = 1e-3

# ── Difference-map thresholds (metres) ────────────────────────────────────────
# Pixels where |diff| <= ZERO_TOL are counted as "zero" (negligible error)
ZERO_TOL         = 0.10   # ± 10 cm → zero
POS_THRESH       = 2.0    # > 0.1 and <= 2.0 → positive (model underestimates)
TOO_POS_THRESH   = 5.0    # > 2.0 and <= 5.0 → too_positive
# > 5.0 → extreme_positive
NEG_THRESH       = -2.0   # < -0.1 and >= -2.0 → negative (model overestimates)
TOO_NEG_THRESH   = -5.0   # < -2.0 and >= -5.0 → too_negative
# < -5.0 → extreme_negative

# ── Overlay visualisation ─────────────────────────────────────────────────────
SCATTER_POINT_SIZE = 6     # point size in overlay scatter plot
OVERLAY_ALPHA      = 0.85  # opacity of difference-map dots

# ── Camera / Focal-length source ─────────────────────────────────────────────
#
# DA3METRIC-LARGE metric conversion (hard-coded per model card):
#   metric_depth = focal * net_output / 300.
# where `focal` is the focal length in pixels — typically avg(fx, fy) from K.
#
# USE_MODEL_FOCAL = True  → use the focal length predicted by the model itself
#                           (preds["focal"] scalar tensor, in pixels).
#                           Falls back to FOCAL_LENGTH_KITTI if the model does
#                           not return a focal key for a given frame.
# USE_MODEL_FOCAL = False → always use the hard-coded KITTI default below.
#
USE_MODEL_FOCAL = True      # ← change to True to use model-predicted focal

# KITTI default: fx = fy ≈ 721.5577  →  avg(fx,fy) = 721.5577
# (camera_02, sequences 2011_09_26 through 2011_10_03)
FOCAL_LENGTH_KITTI = 721.5577

# ── DA3 model identifier ──────────────────────────────────────────────────────
MODEL_ID = "depth-anything/DA3METRIC-LARGE"

# ── Output directories ────────────────────────────────────────────────────────
MAIN_OUTPUT_DIR  = "/STORAGE/Arka/a_MDE_Code/1_Anaysis/Results/DA3_TREND_ANALYSIS_FINAL_modal_focal"
OVERLAY_DIR      = os.path.join(MAIN_OUTPUT_DIR, "overlays")       # PNG overlays
FRAME_CSV_PATH   = os.path.join(MAIN_OUTPUT_DIR, "frame_metrics.csv")   # per-frame
MASK_CSV_PATH    = os.path.join(MAIN_OUTPUT_DIR, "mask_metrics.csv")    # per-mask
PLOT_DIR         = os.path.join(MAIN_OUTPUT_DIR, "analysis_plots")

for d in [MAIN_OUTPUT_DIR, OVERLAY_DIR, PLOT_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Resume support ────────────────────────────────────────────────────────────
SKIP_INFERENCE = False   # set True to skip inference and jump straight to analysis

print("Configuration loaded.")

Configuration loaded.


## 2 · Logging

In [3]:
logging.basicConfig(
    level   = logging.INFO,
    format  = "%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt = "%H:%M:%S",
)
log = logging.getLogger("da3_trend")
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("urllib3").setLevel(logging.ERROR)

## 3 · Metric & Helper Functions

In [4]:
# ── Standard depth error metrics ──────────────────────────────────────────────
def compute_depth_errors(gt: np.ndarray, pred: np.ndarray, mask: np.ndarray) -> dict:
    """KITTI / Monodepth2 standard depth metrics (masked pixels only). NO scaling."""
    gt_v   = gt[mask].astype(np.float64)
    pred_v = pred[mask].astype(np.float64)
    pred_v = np.maximum(pred_v, 1e-8)
    thresh   = np.maximum(gt_v / pred_v, pred_v / gt_v)
    rmse     = float(np.sqrt(np.mean((gt_v - pred_v) ** 2)))
    mae      = float(np.mean(np.abs(gt_v - pred_v)))
    abs_rel  = float(np.mean(np.abs(gt_v - pred_v) / gt_v))
    sq_rel   = float(np.mean(((gt_v - pred_v) ** 2) / gt_v))
    rmse_log = float(np.sqrt(np.mean((np.log(gt_v) - np.log(pred_v)) ** 2)))
    log10    = float(np.mean(np.abs(np.log10(gt_v) - np.log10(pred_v))))
    delta1   = float(np.mean(thresh < 1.25))
    delta2   = float(np.mean(thresh < 1.25 ** 2))
    delta3   = float(np.mean(thresh < 1.25 ** 3))
    mean_err = float(np.mean(gt_v - pred_v))     # signed bias
    return dict(n_pts=int(mask.sum()), rmse=rmse, mae=mae, abs_rel=abs_rel,
                sq_rel=sq_rel, rmse_log=rmse_log, log10=log10,
                delta1=delta1, delta2=delta2, delta3=delta3,
                mean_error=mean_err,
                lidar_mean=float(np.mean(gt_v)), pred_mean=float(np.mean(pred_v)))


# ── Point-count categorisation ─────────────────────────────────────────────────
def count_diff_categories(diff: np.ndarray, mask: np.ndarray) -> dict:
    """
    diff = lidar - pred  (positive → model underestimates; negative → overestimates)
    Returns counts for each category.
    """
    d = diff[mask]
    n_total          = int(mask.sum())
    n_zero           = int(np.sum(np.abs(d) <= ZERO_TOL))
    n_positive       = int(np.sum((d >  ZERO_TOL) & (d <= POS_THRESH)))
    n_too_positive   = int(np.sum((d >  POS_THRESH) & (d <= TOO_POS_THRESH)))
    n_extreme_pos    = int(np.sum(d  >  TOO_POS_THRESH))
    n_negative       = int(np.sum((d < -ZERO_TOL) & (d >= NEG_THRESH)))
    n_too_negative   = int(np.sum((d <  NEG_THRESH) & (d >= TOO_NEG_THRESH)))
    n_extreme_neg    = int(np.sum(d  <  TOO_NEG_THRESH))
    return dict(
        n_valid_pts      = n_total,
        n_zero           = n_zero,
        n_positive       = n_positive,
        n_too_positive   = n_too_positive,
        n_extreme_pos    = n_extreme_pos,
        n_negative       = n_negative,
        n_too_negative   = n_too_negative,
        n_extreme_neg    = n_extreme_neg,
        pct_zero         = 100.0 * n_zero         / max(n_total, 1),
        pct_positive     = 100.0 * n_positive     / max(n_total, 1),
        pct_too_positive = 100.0 * n_too_positive / max(n_total, 1),
        pct_extreme_pos  = 100.0 * n_extreme_pos  / max(n_total, 1),
        pct_negative     = 100.0 * n_negative     / max(n_total, 1),
        pct_too_negative = 100.0 * n_too_negative / max(n_total, 1),
        pct_extreme_neg  = 100.0 * n_extreme_neg  / max(n_total, 1),
        diff_mean   = float(np.mean(d)),
        diff_std    = float(np.std(d)),
        diff_median = float(np.median(d)),
    )


# ── Diverging colormap: white=0, red=positive, blue=negative ──────────────────
def build_diff_cmap():
    """
    Custom diverging colormap:
      deep-blue → light-blue → white → light-red → deep-red
    Centred on 0 (white).
    """
    colors_list = [
        (0.0,  "#7f0000"),   
        (0.25, "#fc8d59"),  
        (0.5,  "#ffffff"),  
        (0.75, "#6baed6"),  
        (1.0,  "#08306b"),   
    ]
    cmap = mcolors.LinearSegmentedColormap.from_list(
        "diff_rw_b",
        [(p, c) for p, c in colors_list]
    )
    return cmap

DIFF_CMAP = build_diff_cmap()


def eigen_crop_mask(shape) -> np.ndarray:
    """Standard Eigen-crop region."""
    h, w = shape[:2]
    m = np.zeros((h, w), dtype=bool)
    top = 44; bot = min(top + 352, h)
    m[top:bot, :] = True
    return m

print("Metric helpers defined.")

Metric helpers defined.


## 4 · DA3METRIC-LARGE Model Loading & Inference

In [5]:
_da3_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std =[0.229, 0.224, 0.225]),
])


def load_da3_model():
    log.info("Loading DA3METRIC-LARGE (%s) …", MODEL_ID)
    mdl = DepthAnything3.from_pretrained(MODEL_ID, cache_dir=CACHE_DIR).to(DEVICE).eval()
    log.info("Model ready on %s.", DEVICE)
    return mdl


def run_da3(mdl, im_np: np.ndarray) -> tuple[np.ndarray, float, str]:
    """
    Run DA3METRIC-LARGE on one uint8 RGB image (H×W×3).

    Metric conversion (hard-coded per model card):
        metric_depth = focal * net_output / 300.
    where focal = avg(fx, fy) in pixels.

    Focal source is controlled by USE_MODEL_FOCAL:
      True  → use preds['intrinsics'] returned by the model (falls back to
               FOCAL_LENGTH_KITTI if the key is absent or non-positive).
      False → always use FOCAL_LENGTH_KITTI.

    Returns
    -------
    depth_m   : float32 ndarray, shape (H, W), metric depth in metres.
    focal_used: float, the focal length (pixels) actually applied.
    focal_src : str, one of 'model' | 'kitti_default' | 'kitti_fallback'.
    """
    orig_h, orig_w = im_np.shape[:2]
    new_h = int(round(orig_h / 14) * 14)
    new_w = int(round(orig_w / 14) * 14)

    img_pil = Image.fromarray(im_np).resize((new_w, new_h), Image.BICUBIC)
    t_img   = _da3_transform(img_pil)
    inp     = t_img.unsqueeze(0).unsqueeze(1).to(DEVICE)   # (1, 1, 3, H, W)

    with torch.no_grad():
        preds = mdl(inp, export_feat_layers=[])

        # ── Extract raw network output (before focal scaling) ──────────────────
        if isinstance(preds, dict):
            net_out = preds["depth"]
        elif isinstance(preds, (tuple, list)):
            net_out = preds[0]
        else:
            net_out = preds

        if not isinstance(net_out, torch.Tensor):
            raise RuntimeError(f"Unexpected depth output type: {type(net_out)}")

        if net_out.dim() == 3:
            net_out = net_out.unsqueeze(1)   # → (B, 1, H', W')

        # ── Determine focal length to use ──────────────────────────────────────
        focal_used: float
        focal_src:  str

        # if USE_MODEL_FOCAL:
        #     # The model may return a predicted focal as preds['focal'].
        #     # It is a scalar tensor (pixels) representing avg(fx, fy).
        #     model_focal = None
        #     if isinstance(preds, dict):
        #         raw_f = preds.get("focal", None)
        #         if raw_f is not None:
        #             f_val = float(raw_f.squeeze()) if isinstance(raw_f, torch.Tensor) \
        #                     else float(raw_f)
        #             if f_val > 0:
        #                 model_focal = f_val
        #     if model_focal is not None:
        #         focal_used = model_focal
        #         focal_src  = "model"
        #     else:
        #         # Model did not return a usable focal — fall back gracefully
        #         focal_used = FOCAL_LENGTH_KITTI
        #         focal_src  = "kitti_fallback"
        #         log.debug("Model focal unavailable; falling back to KITTI default.")
        # else:
        #     focal_used = FOCAL_LENGTH_KITTI
        #     focal_src  = "kitti_default"
        if USE_MODEL_FOCAL:
            model_focal = None
            if isinstance(preds, dict):
                # Look for the new intrinsics matrix [B, 3, 3] or [3, 3]
                intrinsics = preds.get("intrinsics", None)
                
                if intrinsics is not None:
                    # Handle batch dimension if present
                    if intrinsics.dim() == 3:
                        fx = intrinsics[0, 0, 0]
                        fy = intrinsics[0, 1, 1]
                    else:
                        fx = intrinsics[0, 0]
                        fy = intrinsics[1, 1]
                    
                    # Calculate average of fx and fy
                    f_val = float((fx + fy) / 2.0)
                    if f_val > 0:
                        model_focal = f_val
                
                # Fallback just in case older DA3 versions use a scalar "focal" key
                elif "focal" in preds:
                    raw_f = preds["focal"]
                    f_val = float(raw_f.squeeze()) if isinstance(raw_f, torch.Tensor) else float(raw_f)
                    if f_val > 0:
                        model_focal = f_val

            if model_focal is not None:
                focal_used = model_focal
                focal_src  = "model"
            else:
                # Model did not return a usable focal — fall back gracefully
                focal_used = FOCAL_LENGTH_KITTI
                focal_src  = "kitti_fallback"
                log.debug("Model focal unavailable; falling back to KITTI default.")
        else:
            focal_used = FOCAL_LENGTH_KITTI
            focal_src  = "kitti_default"
            
        # ── Hard-coded metric conversion from model card ───────────────────────
        #   metric_depth = focal * net_output / 300.
        depth_m = (focal_used * net_out) / 300.0

        # ── Upsample back to original resolution ──────────────────────────────
        depth_m = F.interpolate(
            depth_m, size=(orig_h, orig_w),
            mode="bilinear", align_corners=False,
        )

    depth_np = depth_m.squeeze().cpu().numpy().astype(np.float32)
    return depth_np, focal_used, focal_src

print("DA3 helpers defined.")

DA3 helpers defined.


## 5 · KITTI Dataset Indexing

In [6]:
def find_lidar_gt(rgb_path: str, split: str) -> str | None:
    """
    Locate the LiDAR ground-truth PNG that corresponds to an RGB image.
    KITTI Eigen layout:
      <split>/<date>/<drive>/image_02/data/<frame>.png
      <split>/<date>/<drive>/proj_depth/groundtruth/image_02/<frame>.png
    """
    rgb_p    = Path(rgb_path)
    frame    = rgb_p.name
    # Walk up: data/ → image_02/ → <drive>/ → <date>/ → <split>/
    drive_dir = rgb_p.parent.parent.parent   # <split>/<date>/<drive>
    gt_path   = drive_dir / "proj_depth" / "groundtruth" / "image_02" / frame
    return str(gt_path) if gt_path.exists() else None


def load_lidar_gt(gt_path: str) -> np.ndarray:
    """
    Load a KITTI LiDAR depth PNG (uint16, depth = pixel / 256.0 metres).
    Returns float32 map; 0.0 = invalid.
    """
    depth = np.array(Image.open(gt_path), dtype=np.float32) / 256.0
    return depth


def index_kitti(use_train: bool = False) -> list:
    """
    Collect all (rgb_path, lidar_gt_path, split) tuples from the dataset.
    Frames without a matching LiDAR GT are skipped with a warning.
    """
    splits = [(TEST_ROOT, "test")]
    if use_train:
        splits.append((TRAIN_ROOT, "train"))

    records = []
    for root, split in splits:
        pattern = os.path.join(root, "*", "*", "image_02", "data", "*.png")
        rgb_files = sorted(glob.glob(pattern))
        log.info("[%s] Found %d RGB images.", split, len(rgb_files))
        for rgb_path in rgb_files:
            gt_path = find_lidar_gt(rgb_path, split)
            if gt_path is None:
                log.debug("No LiDAR GT for %s — skipped.", rgb_path)
                continue
            records.append({"rgb_path": rgb_path, "gt_path": gt_path, "split": split})
    log.info("Total frames with GT: %d", len(records))
    return records

print("Dataset helpers defined.")

Dataset helpers defined.


## 6 · Difference-Map Overlay Visualisation

In [7]:
def make_overlay_figure(
    rgb_np:   np.ndarray,   # uint8 H×W×3
    diff_map: np.ndarray,   # float32 H×W  (LiDAR − pred, valid LiDAR pixels only)
    lidar_mask: np.ndarray, # bool   H×W  (True where LiDAR is valid)
    clip_label: str,
    frame_name: str,
    vmax_abs: float = None,
) -> plt.Figure:
    """
    Scatter the difference values on top of the original RGB image.
    Colormap: white=0, deep-red=positive, deep-blue=negative.
    Returns the matplotlib Figure.
    """
    ys, xs = np.where(lidar_mask)
    vals   = diff_map[lidar_mask]

    if vmax_abs is None:
        vmax_abs = max(float(np.percentile(np.abs(vals), 98)), 0.1)

    norm   = mcolors.TwoSlopeNorm(vmin=-vmax_abs, vcenter=0.0, vmax=vmax_abs)

    h, w   = rgb_np.shape[:2]
    dpi    = 100
    fig_w  = w / dpi + 1.5   # extra width for colorbar
    fig_h  = h / dpi

    fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=dpi)
    ax.imshow(rgb_np, aspect="auto")
    sc = ax.scatter(
        xs, ys,
        c=vals, cmap=DIFF_CMAP, norm=norm,
        s=SCATTER_POINT_SIZE, alpha=OVERLAY_ALPHA,
        linewidths=0, rasterized=True,
    )
    cbar = fig.colorbar(sc, ax=ax, fraction=0.03, pad=0.01)
    cbar.set_label("LiDAR − Pred  (m)", fontsize=9)
    ax.set_title(f"{frame_name}  |  {clip_label}", fontsize=9, pad=4)
    ax.axis("off")
    plt.tight_layout(pad=0.5)
    return fig


def save_overlay(
    rgb_np:    np.ndarray,
    diff_map:  np.ndarray,
    lidar_mask: np.ndarray,
    clip_label: str,
    frame_name: str,
    out_dir:   str,
    vmax_abs:  float = None,
) -> str:
    """Generate + save overlay PNG. Returns saved path."""
    fig  = make_overlay_figure(rgb_np, diff_map, lidar_mask,
                               clip_label, frame_name, vmax_abs)
    fname = f"{frame_name}__{clip_label}.png"
    out   = os.path.join(out_dir, fname)
    fig.savefig(out, bbox_inches="tight", dpi=100)
    plt.close(fig)
    return out

print("Overlay helpers defined.")

Overlay helpers defined.


## 7 · Pre-Generated Binary Mask Loading

In [8]:
def get_mask_dir(rgb_path: str) -> str:
    """
    Map an RGB image path to its pre-generated mask directory.
    KITTI_ROOT / <split> / <date> / <drive> / image_02/data / <frame>.png
         →
    MASK_ROOT  / <split> / <date> / <drive> / image_02/data / <frame_stem> /
    """
    frame_stem   = os.path.splitext(os.path.basename(rgb_path))[0]
    rel_data_dir = os.path.relpath(os.path.dirname(rgb_path), KITTI_ROOT)
    return os.path.join(MASK_ROOT, rel_data_dir, frame_stem)


def load_masks_for_frame(mask_dir: str) -> list:
    """
    Load all pre-generated instance masks for one frame.
    Returns list of dicts: uid, super_cat, sub_cat, score, mask (bool H×W).
    """
    mask_dir_path = Path(mask_dir)
    if not mask_dir_path.exists():
        return []

    frame_stem    = mask_dir_path.name
    manifest_path = mask_dir_path / f"{frame_stem}_manifest.json"

    if manifest_path.exists():
        with open(manifest_path) as fh:
            records = json.load(fh)
        instances = []
        for rec in records:
            if float(rec.get("score", 1.0)) < MIN_MASK_SCORE:
                continue
            mf = mask_dir_path / rec["mask_file"]
            if not mf.exists():
                continue
            mask_np = np.array(Image.open(mf).convert("L")) > 127
            instances.append({
                "uid":       rec["uid"],
                "local_idx": rec["local_idx"],
                "super_cat": rec["super_cat"],
                "sub_cat":   rec["sub_cat"],
                "score":     float(rec["score"]),
                "mask":      mask_np,
            })
        return instances

    # Fallback: parse filenames
    instances = []
    for mf in sorted(mask_dir_path.glob("*_mask.png")):
        stem = mf.stem  # e.g. 0000000005_vehicle_vehicle_headlights_0000_a3f2c1b0_0.923_mask
        if "_mask" in stem:
            stem = stem.rsplit("_mask", 1)[0]
        parts = stem.split("_")
        if len(parts) < 6:
            continue
        try:
            score = float(parts[-1])
            uid   = parts[-2]
            # super_cat is parts[1], sub_cat is parts[2:-3] joined
            super_cat = parts[1]
            sub_cat   = "_".join(parts[2:-3]) if len(parts) > 5 else parts[2]
        except (ValueError, IndexError):
            continue
        if score < MIN_MASK_SCORE:
            continue
        mask_np = np.array(Image.open(mf).convert("L")) > 127
        instances.append({
            "uid": uid, "local_idx": 0,
            "super_cat": super_cat, "sub_cat": sub_cat,
            "score": score, "mask": mask_np,
        })
    return instances

print("Mask loading helpers defined.")

Mask loading helpers defined.


## 8 · Main Processing Loop

In [9]:
def process_frame(
    mdl,
    rec: dict,
    frame_rows: list,
    mask_rows:  list,
):
    """
    Process a single KITTI frame:
      1. Run DA3 inference → raw pred depth
      2. For each clip variant: mask + clip, compute diff map, save overlay PNG,
         record frame-level metrics + point counts in frame_rows
      3. Load binary instance masks; for each mask × each clip variant:
         compute metrics + point counts → mask_rows
    """
    rgb_path  = rec["rgb_path"]
    gt_path   = rec["gt_path"]
    split     = rec["split"]
    frame_name = Path(rgb_path).stem   # e.g. 0000000000

    # Derive a short drive key for the filename
    # <split>/<date>/<drive>/image_02/data/<frame>.png
    p_parts  = Path(rgb_path).parts
    try:
        img_idx  = p_parts.index("image_02")
        drive_id = p_parts[img_idx - 1]    # e.g. 2011_09_26_drive_0002_sync
        date_id  = p_parts[img_idx - 2]    # e.g. 2011_09_26
    except ValueError:
        drive_id = "unknown_drive"
        date_id  = "unknown_date"

    unique_id = f"{split}__{date_id}__{drive_id}__{frame_name}"
    frame_overlay_dir = os.path.join(OVERLAY_DIR, split, date_id, drive_id)
    os.makedirs(frame_overlay_dir, exist_ok=True)

    # ── Load RGB image ─────────────────────────────────────────────────────────
    rgb_np = np.array(Image.open(rgb_path).convert("RGB"), dtype=np.uint8)
    H, W   = rgb_np.shape[:2]

    # ── Load LiDAR GT ──────────────────────────────────────────────────────────
    gt_full = load_lidar_gt(gt_path)   # float32 H×W
    # Resize GT to match image if needed (GT might be smaller)
    if gt_full.shape != (H, W):
        gt_pil  = Image.fromarray(gt_full).resize((W, H), Image.NEAREST)
        gt_full = np.array(gt_pil, dtype=np.float32)

    lidar_valid_mask = gt_full > MIN_DEPTH   # True where LiDAR data exists

    # ── DA3 inference (raw, no scaling) ────────────────────────────────────────
    # run_da3 returns (depth_metres, focal_used, focal_src)
    # Formula applied inside: metric_depth = focal * net_output / 300.
    pred_raw, focal_used, focal_src = run_da3(mdl, rgb_np)

    # Determine vmax_abs for consistent colorbar across all three clip variants
    # (use 98th percentile of absolute diff at no-clip to anchor the scale)
    diff_for_scale = gt_full - pred_raw
    if lidar_valid_mask.sum() > 0:
        vmax_abs_global = float(np.percentile(
            np.abs(diff_for_scale[lidar_valid_mask]), 98
        ))
        vmax_abs_global = max(vmax_abs_global, 0.5)
    else:
        vmax_abs_global = 10.0

    # ── Per-clip-variant processing ────────────────────────────────────────────
    for clip_label, max_d in CLIP_VARIANTS:
        # Clip predicted depth
        pred_clipped = pred_raw.copy()
        pred_clipped = np.maximum(pred_clipped, MIN_DEPTH)
        if max_d is not None:
            pred_clipped = np.minimum(pred_clipped, max_d)

        # The evaluation mask: only pixels where LiDAR is valid
        # (and within the clipped range, to be consistent with the GT)
        if max_d is not None:
            eval_mask = lidar_valid_mask & (gt_full <= max_d)
        else:
            eval_mask = lidar_valid_mask.copy()

        if eval_mask.sum() == 0:
            log.debug("No valid LiDAR pts for %s / %s — skip.", unique_id, clip_label)
            continue

        # Difference map: LiDAR − pred  (positive = model underestimates depth)
        diff_map = gt_full - pred_clipped

        # ── Save overlay PNG ───────────────────────────────────────────────────
        save_overlay(
            rgb_np     = rgb_np,
            diff_map   = diff_map,
            lidar_mask = eval_mask,
            clip_label = clip_label,
            frame_name = frame_name,
            out_dir    = frame_overlay_dir,
            vmax_abs   = vmax_abs_global,
        )

        # ── Frame-level metrics ────────────────────────────────────────────────
        err_stats   = compute_depth_errors(gt_full, pred_clipped, eval_mask)
        diff_counts = count_diff_categories(diff_map, eval_mask)

        frame_rows.append({
            "unique_id":   unique_id,
            "split":       split,
            "date":        date_id,
            "drive":       drive_id,
            "frame":       frame_name,
            "clip_variant": clip_label,
            "max_depth_clip": max_d if max_d else "none",
            "focal_used":  focal_used,
            "focal_src":   focal_src,
            **err_stats,
            **diff_counts,
        })

        # ── Per-binary-mask analysis (same clip variant) ───────────────────────
        mask_dir    = get_mask_dir(rgb_path)
        instances   = load_masks_for_frame(mask_dir)

        for inst in instances:
            inst_mask = inst["mask"]
            # Resize instance mask if needed
            if inst_mask.shape != (H, W):
                inst_mask_pil = Image.fromarray(inst_mask.astype(np.uint8) * 255)
                inst_mask     = np.array(
                    inst_mask_pil.resize((W, H), Image.NEAREST)
                ) > 127

            combined_mask = eval_mask & inst_mask
            if combined_mask.sum() < 5:
                continue

            m_err    = compute_depth_errors(gt_full, pred_clipped, combined_mask)
            m_counts = count_diff_categories(diff_map, combined_mask)

            mask_rows.append({
                "unique_id":    unique_id,
                "split":        split,
                "date":         date_id,
                "drive":        drive_id,
                "frame":        frame_name,
                "clip_variant": clip_label,
                "max_depth_clip": max_d if max_d else "none",
                "focal_used":   focal_used,
                "focal_src":    focal_src,
                "uid":          inst["uid"],
                "super_cat":    inst["super_cat"],
                "sub_cat":      inst["sub_cat"],
                "mask_score":   inst["score"],
                **m_err,
                **m_counts,
            })


def run_pipeline(mdl, frame_list: list):
    """Main loop: process all frames and write CSVs."""
    frame_rows: list = []
    mask_rows:  list = []

    # Resume: skip already-processed frames
    done_ids = set()
    if os.path.exists(FRAME_CSV_PATH):
        df_existing = pd.read_csv(FRAME_CSV_PATH)
        done_ids    = set(df_existing["unique_id"].unique())
        frame_rows  = df_existing.to_dict("records")
        log.info("Resuming: %d frames already processed.", len(done_ids))

    done_mask_ids = set()
    if os.path.exists(MASK_CSV_PATH):
        df_m_existing = pd.read_csv(MASK_CSV_PATH)
        done_mask_ids = set(df_m_existing["unique_id"].unique())
        mask_rows     = df_m_existing.to_dict("records")

    for rec in tqdm(frame_list, desc="Processing frames"):
        p      = Path(rec["rgb_path"])
        fn     = p.stem
        try:
            img_idx  = p.parts.index("image_02")
            drive_id = p.parts[img_idx - 1]
            date_id  = p.parts[img_idx - 2]
        except ValueError:
            drive_id = date_id = "unknown"
        uid = f"{rec['split']}__{date_id}__{drive_id}__{fn}"

        if uid in done_ids:
            continue

        try:
            process_frame(mdl, rec, frame_rows, mask_rows)
        except Exception as e:
            log.error("Error on %s: %s", uid, e)
            continue

        # Checkpoint every 50 frames
        if len(frame_rows) % 50 == 0:
            pd.DataFrame(frame_rows).to_csv(FRAME_CSV_PATH, index=False)
            pd.DataFrame(mask_rows ).to_csv(MASK_CSV_PATH,  index=False)

    pd.DataFrame(frame_rows).to_csv(FRAME_CSV_PATH, index=False)
    pd.DataFrame(mask_rows ).to_csv(MASK_CSV_PATH,  index=False)
    log.info("CSVs saved:\n  %s\n  %s", FRAME_CSV_PATH, MASK_CSV_PATH)

print("Pipeline functions defined.")

Pipeline functions defined.


## 9 · Run Inference

In [10]:
if not SKIP_INFERENCE:
    log.info("STEP 1/2: Indexing KITTI Eigen-Split …")
    frame_list = index_kitti(use_train=USE_TRAIN)

    log.info("STEP 2/2: Loading DA3METRIC-LARGE …")
    da3_mdl = load_da3_model()

    log.info("Running pipeline on %d frames …", len(frame_list))
    run_pipeline(da3_mdl, frame_list)
else:
    log.info("Skipping inference — loading CSVs from disk.")

log.info("Loading CSVs …")
df_frame = pd.read_csv(FRAME_CSV_PATH)
df_mask  = pd.read_csv(MASK_CSV_PATH)
log.info("Frame CSV: %d rows | Mask CSV: %d rows", len(df_frame), len(df_mask))
df_frame.head(3)

09:38:50  INFO      STEP 1/2: Indexing KITTI Eigen-Split …
09:38:50  INFO      [test] Found 697 RGB images.
09:38:50  INFO      Total frames with GT: 652
09:38:50  INFO      STEP 2/2: Loading DA3METRIC-LARGE …
09:38:50  INFO      Loading DA3METRIC-LARGE (depth-anything/DA3METRIC-LARGE) …
09:38:51  INFO      HTTP Request: HEAD https://huggingface.co/depth-anything/DA3METRIC-LARGE/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
09:38:51  INFO      HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/depth-anything/DA3METRIC-LARGE/4010e39f3634a45bc60553321fb49fb760bd594e/config.json "HTTP/1.1 200 OK"


[INFO ] using MLP layer as FFN


09:38:53  INFO      HTTP Request: HEAD https://huggingface.co/depth-anything/DA3METRIC-LARGE/resolve/main/model.safetensors "HTTP/1.1 302 Found"
09:38:53  WARNING   Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
09:38:55  INFO      Model ready on cuda.
09:38:55  INFO      Running pipeline on 652 frames …


Processing frames:   0%|          | 0/652 [00:00<?, ?it/s]

10:27:13  INFO      CSVs saved:
  /STORAGE/Arka/a_MDE_Code/1_Anaysis/Results/DA3_TREND_ANALYSIS_FINAL_modal_focal/frame_metrics.csv
  /STORAGE/Arka/a_MDE_Code/1_Anaysis/Results/DA3_TREND_ANALYSIS_FINAL_modal_focal/mask_metrics.csv
10:27:13  INFO      Loading CSVs …
10:27:13  INFO      Frame CSV: 1956 rows | Mask CSV: 35574 rows


,unique_id,split,date,drive,frame,clip_variant,max_depth_clip,focal_used,focal_src,n_pts,...,pct_zero,pct_positive,pct_too_positive,pct_extreme_pos,pct_negative,pct_too_negative,pct_extreme_neg,diff_mean,diff_std,diff_median
0,test__2011_09_26__2011_09_26_drive_0002_sync__...,test,2011_09_26,2011_09_26_drive_0002_sync,6,clip80,80.0,721.5577,kitti_fallback,99002,...,5.876649,62.277530,8.082665,3.132260,17.478435,2.129250,1.023212,0.998100,2.341050,1.358921
1,test__2011_09_26__2011_09_26_drive_0002_sync__...,test,2011_09_26,2011_09_26_drive_0002_sync,6,clip40,40.0,721.5577,kitti_fallback,93007,...,6.241466,65.993957,7.911232,0.640812,17.727698,1.287000,0.197835,0.959910,1.291265,1.354769
2,test__2011_09_26__2011_09_26_drive_0002_sync__...,test,2011_09_26,2011_09_26_drive_0002_sync,6,noclip,none,721.5577,kitti_fallback,99002,...,5.876649,62.277530,8.082665,3.132260,17.477425,2.118139,1.035333,0.997311,2.343628,1.358921


## 10 · Plotting Utilities

In [11]:
sns.set_theme(style="whitegrid", context="paper", font_scale=1.15)

CLIP_PALETTE = {"clip80": "#e66101", "clip40": "#5e3c99", "noclip": "#1b7837"}
CLIP_ORDER   = ["clip80", "clip40", "noclip"]

DIFF_CATS = [
    "n_extreme_pos", "n_too_positive", "n_positive",
    "n_zero",
    "n_negative", "n_too_negative", "n_extreme_neg",
]
DIFF_CAT_LABELS = {
    "n_extreme_pos":  "Extreme +",
    "n_too_positive": "Too +",
    "n_positive":     "Positive",
    "n_zero":         "~Zero",
    "n_negative":     "Negative",
    "n_too_negative": "Too −",
    "n_extreme_neg":  "Extreme −",
}
# DIFF_CAT_COLORS = [
#     "#7f0000", "#d73027", "#fc8d59",
#     "#f7f7f7",
#     "#91bfdb", "#4575b4", "#08306b",
# ]

DIFF_CAT_COLORS = [
    "#08306b", "#4575b4", "#91bfdb", # Now Blues (Positive / Under-estimation)
    "#f7f7f7",                       # Zero / Neutral
    "#fc8d59", "#d73027", "#7f0000", # Now Reds (Negative / Over-estimation)
]


# def true_rmse(df_sub: pd.DataFrame) -> float:
#     """Pixel-weighted RMSE: sqrt(sum(RMSE_i^2 * n_pts_i) / sum(n_pts_i)).

#     Correctly aggregates per-mask RMSEs into one overall figure by weighting
#     each mask's squared error by its pixel count.  Tiny, distant objects with
#     high RMSE no longer dominate the average.

#     Requires columns 'rmse' and 'n_pts' in df_sub.
#     """
#     valid = df_sub[["rmse", "n_pts"]].dropna()
#     if valid.empty or valid["n_pts"].sum() == 0:
#         return float("nan")
#     total_sse = (valid["rmse"] ** 2 * valid["n_pts"]).sum()
#     total_pts  = valid["n_pts"].sum()
#     return float(np.sqrt(total_sse / total_pts))

def true_rmse(df_sub: pd.DataFrame, rmse_col: str = "rmse") -> float:
    """Pixel-weighted RMSE: sqrt(sum(RMSE_i^2 * n_pts_i) / sum(n_pts_i)).

    Correctly aggregates per-mask RMSEs into one overall figure by weighting
    each mask's squared error by its pixel count.  Tiny, distant objects with
    high RMSE no longer dominate the average.

    Requires columns `rmse_col` and 'n_pts' in df_sub.
    """
    valid = df_sub[[rmse_col, "n_pts"]].dropna()
    if valid.empty or valid["n_pts"].sum() == 0:
        return float("nan")
    total_sse = (valid[rmse_col] ** 2 * valid["n_pts"]).sum()
    total_pts = valid["n_pts"].sum()
    return float(np.sqrt(total_sse / total_pts))

def _save(fig: plt.Figure, fname: str, dpi: int = 130) -> None:
    path = os.path.join(PLOT_DIR, fname)
    fig.savefig(path, bbox_inches="tight", dpi=dpi)
    plt.close(fig)
    log.info("Saved: %s", fname)

print("Plot utilities defined.")

Plot utilities defined.


## 11 · Frame-Level Analysis Plots

In [12]:
# ── Plot 01: Summary table — all terminal sections rendered as PNG ─────────────
def plot_summary_table(df_frame: pd.DataFrame, df_mask: pd.DataFrame) -> None:
    """
    Render all four terminal-summary sections into a single 01_summary_table.png:
      1. Frame-Level Metrics (all clip variants)
      2. Mask-Level Metrics — clip80 per category
      3. Mask-Level Overall True RMSE — all clip variants
      4. Diff-Category % — frame level, clip80
    """
    HDR1 = "#2c3e50"   # dark navy
    HDR2 = "#1a5276"   # medium blue
    HDR3 = "#154360"   # deep blue
    HDR4 = "#1e8449"   # green
    ODD  = "#f0f4f8"
    EVEN = "white"
    FS   = 10

    def _style(tbl, n_rows, n_cols, hdr_color):
        tbl.auto_set_font_size(False)
        tbl.set_fontsize(FS)
        tbl.scale(1.0, 1.65)
        for j in range(n_cols):
            tbl[0, j].set_facecolor(hdr_color)
            tbl[0, j].set_text_props(color="white", fontweight="bold")
        for i in range(1, n_rows + 1):
            bg = ODD if i % 2 == 1 else EVEN
            for j in range(n_cols):
                tbl[i, j].set_facecolor(bg)

    # ── Table 1: Frame-level ───────────────────────────────────────────────────
    rows1 = []
    for clip in CLIP_ORDER:
        sub = df_frame[df_frame["clip_variant"] == clip]
        if sub.empty: continue
        rows1.append([clip, f"{len(sub):,}",
                      f"{true_rmse(sub, 'rmse'):.4f}",
                      f"{sub['mae'].mean():.4f}",
                      f"{sub['abs_rel'].mean():.4f}",
                      f"{sub['delta1'].mean():.4f}",
                      f"{sub['mean_error'].mean():.4f}"])
    cols1 = ["Clip", "#Frames", "RMSE (m)", "MAE (m)", "Abs Rel", "\u03b4<1.25", "Bias (m)"]

    # ── Table 2: Mask-level clip80 per category ───────────────────────────────
    rows2 = []
    if not df_mask.empty:
        df80 = df_mask[df_mask["clip_variant"] == "clip80"]
        for cat in sorted(df80["super_cat"].dropna().unique()):
            sub = df80[df80["super_cat"] == cat]
            if sub.empty: continue
            rows2.append([cat, f"{len(sub):,}",
                          f"{true_rmse(sub, 'rmse'):.4f}",
                          f"{sub['mae'].mean():.4f}",
                          f"{sub['abs_rel'].mean():.4f}",
                          f"{sub['delta1'].mean():.4f}",
                          f"{sub['mean_error'].mean():.4f}"])
    cols2 = ["Category", "#Masks", "RMSE (m)", "MAE (m)", "Abs Rel", "\u03b4<1.25", "Bias (m)"]

    # ── Table 3: Overall mask-level, all clip variants ────────────────────────
    rows3 = []
    if not df_mask.empty:
        for clip in ["clip80", "clip40", "noclip"]:
            sub = df_mask[df_mask["clip_variant"] == clip]
            if sub.empty: continue
            rows3.append([clip, f"{len(sub):,}",
                          f"{true_rmse(sub, 'rmse'):.4f}",
                          f"{sub['mae'].mean():.4f}",
                          f"{sub['abs_rel'].mean():.4f}",
                          f"{sub['delta1'].mean():.4f}"])
    cols3 = ["Clip Variant", "#Masks", "True RMSE (m)", "Avg MAE", "Avg Abs Rel", "Avg \u03b4<1.25"]

    # ── Table 4: Diff-category % (clip80, frame-level) ───────────────────────
    pct_col_names = [
        ("pct_extreme_pos",   "Extreme Pos"),
        ("pct_too_positive",  "Too Positive"),
        ("pct_positive",      "Positive"),
        ("pct_zero",          "Zero"),
        ("pct_negative",      "Negative"),
        ("pct_too_negative",  "Too Negative"),
        ("pct_extreme_neg",   "Extreme Neg"),
    ]
    df80f = df_frame[df_frame["clip_variant"] == "clip80"]
    rows4 = [[lbl, f"{df80f[c].mean():.2f}%"]
             for c, lbl in pct_col_names if c in df80f.columns]
    cols4 = ["Diff Category", "Mean %  (clip80 frames)"]

    # ── Figure layout ─────────────────────────────────────────────────────────
    n_cat  = max(len(rows2), 1)
    n_diff = max(len(rows4), 1)
    # height_ratios: one unit ≈ one data row + a little padding
    hr = [len(rows1), n_cat, len(rows3), n_diff]
    fig = plt.figure(figsize=(16, 2.5 + 0.55 * sum(hr)))
    gs  = fig.add_gridspec(4, 1, hspace=0.55, height_ratios=hr)

    focal_src = "MODEL-PREDICTED" if USE_MODEL_FOCAL else "KITTI DEFAULT"
    fig.suptitle(
        f"DA3METRIC-LARGE \u2014 Trend Analysis Summary  (Unscaled Depths Only)\n"
        f"Focal source: {focal_src}  |  metric_depth = focal \u00d7 net_output / 300",
        fontsize=13, fontweight="bold", y=1.02
    )

    # Table 1
    ax1 = fig.add_subplot(gs[0]); ax1.axis("off")
    ax1.set_title("Frame-Level Metrics  (pixel-weighted True RMSE)",
                  fontsize=11, fontweight="bold", loc="left", pad=5, color=HDR1)
    if rows1:
        t1 = ax1.table(cellText=rows1, colLabels=cols1,
                       cellLoc="center", loc="center", bbox=[0, 0, 1, 1])
        _style(t1, len(rows1), len(cols1), HDR1)

    # Table 2
    ax2 = fig.add_subplot(gs[1]); ax2.axis("off")
    ax2.set_title("Mask-Level Metrics \u2014 clip80  (pixel-weighted True RMSE per category)",
                  fontsize=11, fontweight="bold", loc="left", pad=5, color=HDR2)
    if rows2:
        t2 = ax2.table(cellText=rows2, colLabels=cols2,
                       cellLoc="center", loc="center", bbox=[0, 0, 1, 1])
        _style(t2, len(rows2), len(cols2), HDR2)

    # Table 3
    ax3 = fig.add_subplot(gs[2]); ax3.axis("off")
    ax3.set_title("Mask-Level Overall True RMSE \u2014 All Clip Variants",
                  fontsize=11, fontweight="bold", loc="left", pad=5, color=HDR3)
    if rows3:
        t3 = ax3.table(cellText=rows3, colLabels=cols3,
                       cellLoc="center", loc="center", bbox=[0, 0, 1, 1])
        _style(t3, len(rows3), len(cols3), HDR3)

    # Table 4
    ax4 = fig.add_subplot(gs[3]); ax4.axis("off")
    ax4.set_title("Diff-Category %  \u2014 Frame Level, clip80 \n (LiDAR - Pred_depth) Positive-> UnderEstimation, Negative-> OverEstimation",
                  fontsize=11, fontweight="bold", loc="left", pad=5, color=HDR4)
    if rows4:
        t4 = ax4.table(cellText=rows4, colLabels=cols4,
                       cellLoc="center", loc="center", bbox=[0, 0, 1, 1])
        _style(t4, len(rows4), len(cols4), HDR4)

    plt.tight_layout()
    _save(fig, "01_summary_table.png", dpi=160)

plot_summary_table(df_frame, df_mask)

10:27:15  INFO      Saved: 01_summary_table.png


In [13]:
# ── Plot 02: RMSE distribution by clip variant ────────────────────────────────
def plot_rmse_distribution(df: pd.DataFrame) -> None:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=False)
    for ax, clip in zip(axes, CLIP_ORDER):
        sub = df[df["clip_variant"] == clip]["rmse"].dropna()
        ax.hist(sub, bins=60, color=CLIP_PALETTE[clip], alpha=0.80, edgecolor="white")
        ax.axvline(sub.mean(),   color="black", ls="--", lw=2.0,
                   label=f"Mean = {sub.mean():.3f} m")
        ax.axvline(sub.median(), color="grey",  ls=":",  lw=2.0,
                   label=f"Median = {sub.median():.3f} m")
        ax.set_title(f"RMSE — {clip}"); ax.set_xlabel("RMSE (m)")
        ax.set_ylabel("Frame count"); ax.legend(fontsize=9)
    fig.suptitle("RMSE Distribution by Depth-Clip Variant (Unscaled)", fontsize=14)
    plt.tight_layout(); _save(fig, "02_rmse_distribution.png")

plot_rmse_distribution(df_frame)

10:27:16  INFO      Saved: 02_rmse_distribution.png


In [14]:
# ── Plot 03: Mean error (bias) distribution — signed ─────────────────────────
def plot_bias_distribution(df: pd.DataFrame) -> None:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=False)
    for ax, clip in zip(axes, CLIP_ORDER):
        sub = df[df["clip_variant"] == clip]["mean_error"].dropna()
        ax.hist(sub, bins=60, color=CLIP_PALETTE[clip], alpha=0.80, edgecolor="white")
        ax.axvline(0, color="black", ls="-", lw=1.5, label="Zero bias")
        ax.axvline(sub.mean(), color="red", ls="--", lw=2.0,
                   label=f"Mean = {sub.mean():.3f} m")
        ax.set_title(f"Signed Bias (LiDAR−Pred) — {clip}")
        ax.set_xlabel("Mean Error (m)"); ax.set_ylabel("Frame count")
        ax.legend(fontsize=9)
    fig.suptitle("Signed Depth Bias Distribution (Unscaled)", fontsize=14)
    plt.tight_layout(); _save(fig, "03_bias_distribution.png")

plot_bias_distribution(df_frame)

10:27:16  INFO      Saved: 03_bias_distribution.png


In [15]:
# ── Plot 04: Point-count stacked bar — frame level ────────────────────────────
def plot_point_count_bars(df: pd.DataFrame, title_suffix: str = "",
                          save_name: str = "04_point_count_bars.png") -> None:
    """
    Show the average percentage breakdown of diff categories per clip variant.
    """
    pct_cols = [f"pct_{c.replace('n_','',1)}" for c in DIFF_CATS]
    # guard: some cols may be absent
    pct_cols = [c for c in pct_cols if c in df.columns]
    clip_vals = []
    for clip in CLIP_ORDER:
        sub = df[df["clip_variant"] == clip]
        if sub.empty: continue
        row = {"clip": clip}
        for pc in pct_cols:
            row[pc] = sub[pc].mean()
        clip_vals.append(row)
    if not clip_vals: return
    dfc = pd.DataFrame(clip_vals).set_index("clip")

    fig, ax = plt.subplots(figsize=(11, 6))
    bottom = np.zeros(len(dfc))
    for col, label, color in zip(pct_cols, list(DIFF_CAT_LABELS.values()), DIFF_CAT_COLORS):
        if col not in dfc.columns: continue
        vals = dfc[col].values
        ax.bar(dfc.index, vals, bottom=bottom, label=label, color=color, edgecolor="white")
        bottom += vals
    ax.set_ylabel("Average % of valid LiDAR pixels")
    ax.set_title(f"Difference Category Distribution — {title_suffix} \nLiDAR-PredDepth (Blue-> UnderEst, Red-> OverEst)", fontsize=13)
    ax.legend(loc="upper right", fontsize=9, ncol=2)
    ax.set_ylim(0, 105); ax.grid(axis="y", alpha=0.3)
    plt.tight_layout(); _save(fig, save_name)

plot_point_count_bars(df_frame, title_suffix="Frame Level",
                      save_name="04_frame_point_count_bars.png")

10:27:17  INFO      Saved: 04_frame_point_count_bars.png


In [16]:
# ── Plot 05: RMSE vs LiDAR mean depth — scatter ───────────────────────────────
# def plot_rmse_vs_depth(df: pd.DataFrame) -> None:
#     fig, axes = plt.subplots(1, 3, figsize=(18, 5))
#     for ax, clip in zip(axes, CLIP_ORDER):
#         sub = df[df["clip_variant"] == clip].dropna(subset=["lidar_mean", "rmse"])
#         ax.scatter(sub["lidar_mean"], sub["rmse"],
#                    c=CLIP_PALETTE[clip], alpha=0.35, s=8)
#         if len(sub) > 5:
#             z = np.polyfit(sub["lidar_mean"], sub["rmse"], 1)
#             xr = np.linspace(sub["lidar_mean"].min(), sub["lidar_mean"].max(), 100)
#             ax.plot(xr, np.polyval(z, xr), "r--", lw=2)
#         ax.set_xlabel("LiDAR Mean Depth (m)"); ax.set_ylabel("RMSE (m)")
#         ax.set_title(f"RMSE vs Depth — {clip}")
#     fig.suptitle("Depth-Dependent RMSE (Unscaled)", fontsize=14)
#     plt.tight_layout(); _save(fig, "05_rmse_vs_depth.png")

# ── Plot 05: RMSE vs LiDAR mean depth — scatter ───────────────────────────────
def plot_rmse_vs_depth(df: pd.DataFrame) -> None:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    max_err = df["mean_error"].abs().max()
    norm = mcolors.TwoSlopeNorm(vmin=-max_err, vcenter=0.0, vmax=max_err)

    for ax, clip in zip(axes, CLIP_ORDER):
        # 1. ADD THIS LINE: Light gray background
        ax.set_facecolor("#777a7cff")
        
        sub = df[df["clip_variant"] == clip].dropna(subset=["lidar_mean", "rmse", "mean_error"])
        
        sc = ax.scatter(
            sub["lidar_mean"], sub["rmse"],
            c=sub["mean_error"], cmap=DIFF_CMAP, norm=norm, 
            alpha=0.8, s=15, edgecolor="none"
        )
        
        if len(sub) > 5:
            z = np.polyfit(sub["lidar_mean"], sub["rmse"], 1)
            xr = np.linspace(sub["lidar_mean"].min(), sub["lidar_mean"].max(), 100)
            ax.plot(xr, np.polyval(z, xr), "k--", lw=2) 
            
        ax.set_xlabel("LiDAR Mean Depth (m)"); ax.set_ylabel("RMSE (m)")
        ax.set_title(f"RMSE vs Depth — {clip}")
        
    cbar = fig.colorbar(sc, ax=axes.ravel().tolist(), fraction=0.02, pad=0.02)
    cbar.set_label("Mean Error: LiDAR − Pred (m)", fontsize=10)

    fig.suptitle("Depth-Dependent RMSE (Blue = Under-est | Red = Over-est)", fontsize=14)
    _save(fig, "05_rmse_vs_depth.png")

plot_rmse_vs_depth(df_frame)

10:27:17  INFO      Saved: 05_rmse_vs_depth.png


In [17]:
# ── Plot 06: Depth-bin analysis — RMSE and bias ───────────────────────────────
def plot_depth_bin_analysis(df: pd.DataFrame) -> None:
    bins   = np.arange(0, 85, 10)
    labels = [f"{int(b)}–{int(b+10)}m" for b in bins[:-1]]
    df2    = df.copy()
    df2["depth_bin"] = pd.cut(df2["lidar_mean"], bins=bins, labels=labels)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    ax = axes[0]
    for clip in CLIP_ORDER:
        sub = df2[df2["clip_variant"] == clip]
        agg = sub.groupby("depth_bin", observed=True)["rmse"].mean()
        ax.plot(agg.index, agg.values, "o-",
                color=CLIP_PALETTE[clip], label=clip, lw=2)
    ax.set_xlabel("GT Depth Bin"); ax.set_ylabel("Mean RMSE (m)")
    ax.set_title("RMSE by Depth Bin")
    ax.tick_params(axis="x", rotation=45)
    ax.legend(); ax.grid(alpha=0.3)

    ax = axes[1]
    for clip in CLIP_ORDER:
        sub = df2[df2["clip_variant"] == clip]
        agg = sub.groupby("depth_bin", observed=True)["mean_error"].mean()
        ax.plot(agg.index, agg.values, "o-",
                color=CLIP_PALETTE[clip], label=clip, lw=2)
    ax.axhline(0, color="black", ls="--", lw=1.5)
    ax.set_xlabel("GT Depth Bin"); ax.set_ylabel("Mean Bias (m)")
    ax.set_title("Signed Bias by Depth Bin")
    ax.tick_params(axis="x", rotation=45)
    ax.legend(); ax.grid(alpha=0.3)

    fig.suptitle("Depth-Bin Analysis — DA3METRIC-LARGE (Unscaled)", fontsize=14)
    plt.tight_layout(); _save(fig, "06_depth_bin_analysis.png")

plot_depth_bin_analysis(df_frame)

10:27:18  INFO      Saved: 06_depth_bin_analysis.png


In [18]:
# # ── Plot 07: Predicted vs LiDAR mean depth — scatter (clip80) ─────────────────
# def plot_pred_vs_lidar_scatter(df: pd.DataFrame) -> None:
#     fig, axes = plt.subplots(1, 3, figsize=(18, 6))
#     for ax, clip in zip(axes, CLIP_ORDER):
#         sub = df[df["clip_variant"] == clip].dropna(subset=["lidar_mean","pred_mean"])
#         lim = max(sub["lidar_mean"].max(), sub["pred_mean"].max()) * 1.05
#         ax.scatter(sub["lidar_mean"], sub["pred_mean"],
#                    c=CLIP_PALETTE[clip], alpha=0.35, s=8)
#         ax.plot([0, lim], [0, lim], "k--", lw=1.5, label="Ideal 1:1")
#         ax.set_xlim(0, lim); ax.set_ylim(0, lim)
#         ax.set_xlabel("LiDAR Mean Depth (m)"); ax.set_ylabel("Pred Mean Depth (m)")
#         ax.set_title(f"Predicted vs GT — {clip}"); ax.legend(fontsize=8)
#     fig.suptitle("Predicted vs LiDAR GT Depth (Frame-Level Means, Unscaled)", fontsize=14)
#     plt.tight_layout(); _save(fig, "07_pred_vs_lidar.png")

# ── Plot 07: Predicted vs LiDAR mean depth — scatter ──────────────────────────
def plot_pred_vs_lidar_scatter(df: pd.DataFrame) -> None:
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    max_err = df["mean_error"].abs().max()
    norm = mcolors.TwoSlopeNorm(vmin=-max_err, vcenter=0.0, vmax=max_err)

    for ax, clip in zip(axes, CLIP_ORDER):
        # 1. ADD THIS LINE: Set a light gray background so white dots are visible
        ax.set_facecolor("#777a7cff") 
        
        sub = df[df["clip_variant"] == clip].dropna(subset=["lidar_mean", "pred_mean", "mean_error"])
        lim = max(sub["lidar_mean"].max(), sub["pred_mean"].max()) * 1.05
        
        sc = ax.scatter(
            sub["lidar_mean"], sub["pred_mean"],
            c=sub["mean_error"], cmap=DIFF_CMAP, norm=norm, 
            alpha=0.8, s=15, edgecolor="none" # Bumped alpha to 0.8 for better visibility
        )
        
        ax.plot([0, lim], [0, lim], "k--", lw=1.5, label="Ideal 1:1")
        ax.set_xlim(0, lim); ax.set_ylim(0, lim)
        ax.set_xlabel("LiDAR Mean Depth (m)"); ax.set_ylabel("Pred Mean Depth (m)")
        ax.set_title(f"Predicted vs GT — {clip}")
        
        # Adding a white background to the legend so it doesn't clash with the gray plot
        ax.legend(fontsize=8, loc="upper left", facecolor="white", framealpha=1.0)
        
    cbar = fig.colorbar(sc, ax=axes.ravel().tolist(), fraction=0.02, pad=0.02)
    cbar.set_label("Mean Error: LiDAR − Pred (m)", fontsize=10)
    
    fig.suptitle("Predicted vs LiDAR GT Depth (Blue = Under-est | Red = Over-est)", fontsize=14)
    _save(fig, "07_pred_vs_lidar.png")

plot_pred_vs_lidar_scatter(df_frame)

10:27:18  INFO      Saved: 07_pred_vs_lidar.png


In [19]:
# ── Plot 08: Metrics comparison bar chart (clip variants) ─────────────────────
def plot_metric_comparison_bars(df: pd.DataFrame) -> None:
    metric_specs = [
        ("RMSE (m)  ↓",  "rmse"),
        ("MAE (m)  ↓",   "mae"),
        ("Abs Rel  ↓",   "abs_rel"),
        ("Sq Rel  ↓",    "sq_rel"),
        ("RMSE log ↓",   "rmse_log"),
        ("δ<1.25  ↑",    "delta1"),
    ]
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    for ax, (metric_label, col) in zip(axes.flatten(), metric_specs):
        vals  = [df[df["clip_variant"] == c][col].mean() for c in CLIP_ORDER]
        colors = [CLIP_PALETTE[c] for c in CLIP_ORDER]
        bars  = ax.bar(CLIP_ORDER, vals, color=colors, edgecolor="white", width=0.55)
        ax.bar_label(bars, fmt="%.4f", padding=3, fontsize=10)
        ax.set_title(metric_label); ax.set_ylabel(metric_label)
        ax.set_ylim(0, max(vals) * 1.25)
        ax.grid(axis="y", alpha=0.35)
    fig.suptitle("Metric Comparison Across Depth-Clip Variants (Unscaled)", fontsize=14, y=1.01)
    plt.tight_layout(); _save(fig, "08_metric_comparison_bars.png")

plot_metric_comparison_bars(df_frame)

10:27:19  INFO      Saved: 08_metric_comparison_bars.png


## 12 · Per-Mask Analysis Plots

In [20]:
# ── Plot 09: RMSE per super-category (clip80 only, for clarity) ───────────────
def plot_rmse_by_category(df: pd.DataFrame) -> None:
    df80   = df[df["clip_variant"] == "clip80"]
    cats   = sorted(df80["super_cat"].dropna().unique())
    if len(cats) < 1: return

    # Pixel-weighted True RMSE per category (no std bars — True RMSE is a
    # single weighted value, not a distribution with meaningful variance)
    means = [true_rmse(df80[df80["super_cat"] == c], "rmse") for c in cats]

    fig, ax = plt.subplots(figsize=(max(10, 2.5 * len(cats)), 6))
    x    = np.arange(len(cats))
    bars = ax.bar(x, means, color="#3182bd", edgecolor="white", width=0.6)
    ax.bar_label(bars, fmt="%.3f", padding=4, fontsize=9)
    ax.set_xticks(x); ax.set_xticklabels(cats, rotation=30, ha="right", fontsize=10)
    ax.set_ylabel("True RMSE (m)  [pixel-weighted]")
    ax.set_title("True RMSE per Super-Category (clip80, Unscaled)", fontsize=13)
    ax.grid(axis="y", alpha=0.35)
    plt.tight_layout(); _save(fig, "09_rmse_by_category.png")

if not df_mask.empty:
    plot_rmse_by_category(df_mask)

10:27:20  INFO      Saved: 09_rmse_by_category.png


In [21]:
# ── Plot 10: RMSE per super-cat across all clip variants ──────────────────────
def plot_rmse_by_cat_clip(df: pd.DataFrame) -> None:
    cats = sorted(df["super_cat"].dropna().unique())
    if len(cats) < 1: return
    x, w = np.arange(len(cats)), 0.24
    fig, ax = plt.subplots(figsize=(max(10, 2.5 * len(cats)), 6))
    for i, clip in enumerate(CLIP_ORDER):
        sub  = df[df["clip_variant"] == clip]
        # Pixel-weighted True RMSE — no error bars (not a distribution average)
        vals = [true_rmse(sub[sub["super_cat"] == c], "rmse") for c in cats]
        off  = (i - 1) * w
        bars = ax.bar(x + off, vals, width=w,
                      color=CLIP_PALETTE[clip], label=clip, edgecolor="white")
        ax.bar_label(bars, fmt="%.2f", fontsize=7, padding=2)
    ax.set_xticks(x); ax.set_xticklabels(cats, rotation=30, ha="right", fontsize=10)
    ax.set_ylabel("True RMSE (m)  [pixel-weighted]")
    ax.set_title("True RMSE per Category × Clip Variant (Unscaled)", fontsize=13)
    ax.legend(); ax.grid(axis="y", alpha=0.35)
    plt.tight_layout(); _save(fig, "10_rmse_by_cat_clip.png")

if not df_mask.empty:
    plot_rmse_by_cat_clip(df_mask)

10:27:20  INFO      Saved: 10_rmse_by_cat_clip.png


In [22]:
# ── Plot 11: Point-count stacked bar — per category (clip80) ──────────────────
def plot_point_count_by_cat(df: pd.DataFrame) -> None:
    df80 = df[df["clip_variant"] == "clip80"]
    cats = sorted(df80["super_cat"].dropna().unique())
    if not cats: return

    pct_cols = [
        "pct_extreme_pos","pct_too_positive","pct_positive",
        "pct_zero",
        "pct_negative","pct_too_negative","pct_extreme_neg",
    ]
    pct_cols = [c for c in pct_cols if c in df80.columns]

    fig, ax = plt.subplots(figsize=(max(10, 2.5 * len(cats)), 6))
    bottom = np.zeros(len(cats))
    for col, label, color in zip(pct_cols, list(DIFF_CAT_LABELS.values()), DIFF_CAT_COLORS):
        if col not in df80.columns: continue
        vals = [df80[df80["super_cat"] == c][col].mean() for c in cats]
        ax.bar(cats, vals, bottom=bottom, label=label, color=color, edgecolor="white")
        bottom += np.array(vals)

    ax.set_ylabel("Average % of masked LiDAR pixels")
    ax.set_title("Difference Category Distribution per Object Category (clip80, Unscaled) \nLiDAR - PredDepth (Blue-> UnderEstimation, Red-> OverEstimation)",
                 fontsize=12)
    ax.set_xticklabels(cats, rotation=30, ha="right", fontsize=10)
    ax.legend(loc="upper right", fontsize=9, ncol=2)
    ax.set_ylim(0, 110); ax.grid(axis="y", alpha=0.3)
    plt.tight_layout(); _save(fig, "11_point_count_by_cat.png")

if not df_mask.empty:
    plot_point_count_by_cat(df_mask)

10:27:21  INFO      Saved: 11_point_count_by_cat.png


In [23]:
# ── Plot 12: Per-category box plots — RMSE (clip80) ───────────────────────────
def plot_category_boxplots(df: pd.DataFrame) -> None:
    df80 = df[df["clip_variant"] == "clip80"]
    if df80.empty: return
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))

    sns.boxplot(ax=axes[0], data=df80, x="super_cat", y="rmse",
                palette="Set2", flierprops={"marker": ".", "alpha": 0.3})
    axes[0].set_xlabel("Super Category"); axes[0].set_ylabel("RMSE (m)")
    axes[0].set_title("RMSE Distribution per Category (clip80)")
    axes[0].tick_params(axis="x", rotation=30)

    sns.boxplot(ax=axes[1], data=df80, x="super_cat", y="mean_error",
                palette="Set2", flierprops={"marker": ".", "alpha": 0.3})
    axes[1].axhline(0, color="black", ls="--", lw=1.5)
    axes[1].set_xlabel("Super Category"); axes[1].set_ylabel("Mean Error (m)")
    axes[1].set_title("Signed Bias per Category (clip80)")
    axes[1].tick_params(axis="x", rotation=30)

    fig.suptitle("Per-Category Error Analysis (Unscaled)", fontsize=14)
    plt.tight_layout(); _save(fig, "12_category_boxplots.png")

if not df_mask.empty:
    plot_category_boxplots(df_mask)

10:27:21  INFO      Saved: 12_category_boxplots.png


In [24]:
# ── Plot 13: Heatmap — True RMSE by super_cat × clip_variant ─────────────────
def plot_cat_clip_heatmap(df: pd.DataFrame) -> None:
    cats  = sorted(df["super_cat"].dropna().unique())
    clips = [c for c in CLIP_ORDER if c in df["clip_variant"].unique()]
    if not cats or not clips: return

    # Build matrix manually — pivot_table cannot handle multi-column
    # weighted aggregation natively
    matrix = {}
    for cat in cats:
        matrix[cat] = {}
        for clip in clips:
            sub = df[(df["super_cat"] == cat) & (df["clip_variant"] == clip)]
            matrix[cat][clip] = true_rmse(sub, "rmse")

    import pandas as _pd
    pivot = _pd.DataFrame(matrix).T[clips]  # rows=cats, cols=clips

    fig, ax = plt.subplots(figsize=(10, max(4, 0.55 * len(pivot))))
    sns.heatmap(pivot, annot=True, fmt=".3f", cmap="YlOrRd",
                linewidths=0.5, ax=ax, cbar_kws={"label": "True RMSE (m)  [pixel-weighted]"})
    ax.set_title("True RMSE — Category × Clip Variant (Unscaled)", fontsize=13)
    ax.set_xlabel("Clip Variant"); ax.set_ylabel("Super Category")
    plt.tight_layout(); _save(fig, "13_cat_clip_heatmap.png")

if not df_mask.empty:
    plot_cat_clip_heatmap(df_mask)

10:27:22  INFO      Saved: 13_cat_clip_heatmap.png


In [25]:
# ── Plot 14: Sub-category True RMSE bar — top-N ──────────────────────────────
def plot_subcat_rmse(df: pd.DataFrame, top_n: int = 20) -> None:
    df80 = df[df["clip_variant"] == "clip80"]
    if df80.empty or "sub_cat" not in df80.columns: return

    # Manual per-sub-category True RMSE — groupby.mean() would give the
    # arithmetic mean of individual RMSEs, not the pixel-weighted RMSE
    sub_cats = df80["sub_cat"].dropna().unique()
    rows = []
    for sc in sub_cats:
        grp = df80[df80["sub_cat"] == sc]
        rows.append({"sub_cat": sc, "true_rmse": true_rmse(grp, "rmse")})
    import pandas as _pd
    agg = (_pd.DataFrame(rows)
             .sort_values("true_rmse", ascending=False)
             .head(top_n)
             .set_index("sub_cat")["true_rmse"])

    overall_mean = agg.mean()  # mean-of-means for reference line only
    fig, ax = plt.subplots(figsize=(14, 6))
    agg.plot.barh(ax=ax, color="#e07b54", edgecolor="white")
    ax.set_xlabel("True RMSE (m)  [pixel-weighted]")
    ax.set_title(f"Top-{top_n} Sub-Categories by True RMSE (clip80, Unscaled)", fontsize=13)
    ax.axvline(overall_mean, color="navy", ls="--", lw=1.5,
               label=f"Mean of top-{top_n} = {overall_mean:.3f} m")
    ax.legend(); ax.grid(axis="x", alpha=0.35)
    plt.tight_layout(); _save(fig, "14_subcat_rmse.png")

if not df_mask.empty:
    plot_subcat_rmse(df_mask)

10:27:22  INFO      Saved: 14_subcat_rmse.png


In [26]:
# ── Plot 15: Diff-category proportions — stacked bar per sub-cat (clip80) ────
def plot_subcat_diff_heatmap(df: pd.DataFrame, top_n: int = 20) -> None:
    df80 = df[df["clip_variant"] == "clip80"]
    if df80.empty: return
    pct_cols = [
        "pct_extreme_pos","pct_too_positive","pct_positive",
        "pct_zero",
        "pct_negative","pct_too_negative","pct_extreme_neg",
    ]
    pct_cols = [c for c in pct_cols if c in df80.columns]
    top_cats = (df80.groupby("sub_cat")["rmse"]
                   .mean().sort_values(ascending=False)
                   .head(top_n).index.tolist())
    sub = df80[df80["sub_cat"].isin(top_cats)]
    pivot = sub.pivot_table(values=pct_cols, index="sub_cat", aggfunc="mean")
    if pivot.empty: return
    pivot = pivot[pct_cols]
    pivot.columns = list(DIFF_CAT_LABELS.values())[:len(pivot.columns)]
    fig, ax = plt.subplots(figsize=(14, max(5, 0.55 * len(pivot))))
    # sns.heatmap(pivot, annot=True, fmt=".1f", cmap="RdBu_r",
    #             linewidths=0.4, ax=ax, vmin=0, vmax=100,
    #             cbar_kws={"label": "% of masked pixels"})
    sns.heatmap(pivot, annot=True, fmt=".1f", cmap="Blues", # Or "viridis", "magma", etc.
                linewidths=0.4, ax=ax, vmin=0, vmax=100,
                cbar_kws={"label": "% of masked pixels"})
    ax.set_title(f"Diff-Category % — Top-{top_n} Sub-Cats (clip80)", fontsize=13)
    ax.set_xlabel("Difference Category"); ax.set_ylabel("Sub-Category")
    plt.tight_layout(); _save(fig, "15_subcat_diff_heatmap.png")

if not df_mask.empty:
    plot_subcat_diff_heatmap(df_mask)

10:27:23  INFO      Saved: 15_subcat_diff_heatmap.png


In [27]:
# # ── Plot 16: Mask-level scatter — RMSE vs LiDAR depth per super-cat ──────────
# def plot_mask_rmse_vs_depth_by_cat(df: pd.DataFrame) -> None:
#     df80 = df[df["clip_variant"] == "clip80"].dropna(subset=["lidar_mean","rmse"])
#     cats = sorted(df80["super_cat"].dropna().unique())
#     n    = len(cats)
#     if n == 0: return
#     cols = min(4, n)
#     rows = math.ceil(n / cols)
#     fig, axes = plt.subplots(rows, cols, figsize=(5.5 * cols, 4.5 * rows))
#     axes_flat = np.array(axes).flatten() if n > 1 else [axes]
#     palette   = sns.color_palette("tab10", n)
#     for ax, cat, color in zip(axes_flat, cats, palette):
#         sub = df80[df80["super_cat"] == cat]
#         ax.scatter(sub["lidar_mean"], sub["rmse"],
#                    c=[color], alpha=0.4, s=10)
#         if len(sub) > 5:
#             z  = np.polyfit(sub["lidar_mean"], sub["rmse"], 1)
#             xr = np.linspace(sub["lidar_mean"].min(), sub["lidar_mean"].max(), 80)
#             ax.plot(xr, np.polyval(z, xr), "r--", lw=1.8)
#         ax.set_title(cat, fontsize=10)
#         ax.set_xlabel("LiDAR depth (m)"); ax.set_ylabel("RMSE (m)")
#     for ax in axes_flat[n:]:
#         ax.set_visible(False)
#     fig.suptitle("Mask RMSE vs LiDAR Depth — per Category (clip80)", fontsize=13, y=1.01)
#     plt.tight_layout(); _save(fig, "16_mask_rmse_vs_depth_by_cat.png")

# ── Plot 16: Mask-level scatter — RMSE vs LiDAR depth per super-cat ──────────
def plot_mask_rmse_vs_depth_by_cat(df: pd.DataFrame) -> None:
    # Ensure we drop rows missing the mean_error column used for coloring
    df80 = df[df["clip_variant"] == "clip80"].dropna(subset=["lidar_mean", "rmse", "mean_error"])
    cats = sorted(df80["super_cat"].dropna().unique())
    n    = len(cats)
    if n == 0: return
    cols = min(4, n)
    rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(5.5 * cols, 4.5 * rows))
    axes_flat = np.array(axes).flatten() if n > 1 else [axes]
    
    # Calculate global max absolute error for this subset to anchor the colorbar at 0
    max_err = df80["mean_error"].abs().max()
    if pd.isna(max_err) or max_err == 0:
        max_err = 0.1
    norm = mcolors.TwoSlopeNorm(vmin=-max_err, vcenter=0.0, vmax=max_err)

    sc = None # To hold the scatter plot reference for the colorbar
    
    for ax, cat in zip(axes_flat, cats):
        # Apply the requested background color
        ax.set_facecolor("#777a7cff")
        
        sub = df80[df80["super_cat"] == cat]
        
        # Color the scatter dots using the DIFF_CMAP based on mean_error
        sc = ax.scatter(
            sub["lidar_mean"], sub["rmse"],
            c=sub["mean_error"], cmap=DIFF_CMAP, norm=norm,
            alpha=0.8, s=15, edgecolor="none"
        )
        
        if len(sub) > 5:
            z  = np.polyfit(sub["lidar_mean"], sub["rmse"], 1)
            xr = np.linspace(sub["lidar_mean"].min(), sub["lidar_mean"].max(), 80)
            # Changed trendline to white so it contrasts against the dark gray background
            ax.plot(xr, np.polyval(z, xr), "k--", lw=1.6)
            
        ax.set_title(cat, fontsize=10)
        ax.set_xlabel("LiDAR depth (m)"); ax.set_ylabel("RMSE (m)")
        
    for ax in axes_flat[n:]:
        ax.set_visible(False)
        
    # Attach the shared colorbar
    # if sc is not None:
    #     cbar = fig.colorbar(sc, ax=axes_flat.tolist(), fraction=0.02, pad=0.02)
    #     cbar.set_label("Mean Error: LiDAR − Pred (m)", fontsize=10)
        
    fig.suptitle("Mask RMSE vs LiDAR Depth — per Category (Blue = Under-est | Red = Over-est)", fontsize=13, y=1.02)
    plt.tight_layout(); _save(fig, "16_mask_rmse_vs_depth_by_cat.png")

if not df_mask.empty:
    plot_mask_rmse_vs_depth_by_cat(df_mask)

10:27:24  INFO      Saved: 16_mask_rmse_vs_depth_by_cat.png


In [28]:
# ── Plot 17: Abs-Rel and delta1 comparison — frame vs category-masked ─────────
def plot_frame_vs_mask_metrics(df_frame: pd.DataFrame, df_mask: pd.DataFrame) -> None:
    if df_mask.empty: return
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Abs Rel
    ax = axes[0]
    for clip in CLIP_ORDER:
        f_val = df_frame[df_frame["clip_variant"] == clip]["abs_rel"].mean()
        m_val = df_mask[df_mask["clip_variant"] == clip]["abs_rel"].mean()
        x = ["Frame-Level", "Mask-Level"]
        ax.plot(x, [f_val, m_val], "o-", color=CLIP_PALETTE[clip],
                label=clip, lw=2, ms=8)
    ax.set_title("Abs Rel: Frame vs Mask Level"); ax.set_ylabel("Abs Rel")
    ax.legend(); ax.grid(alpha=0.3)

    # delta1
    ax = axes[1]
    for clip in CLIP_ORDER:
        f_val = df_frame[df_frame["clip_variant"] == clip]["delta1"].mean()
        m_val = df_mask[df_mask["clip_variant"] == clip]["delta1"].mean()
        ax.plot(["Frame-Level", "Mask-Level"], [f_val, m_val], "o-",
                color=CLIP_PALETTE[clip], label=clip, lw=2, ms=8)
    ax.set_title("δ<1.25: Frame vs Mask Level"); ax.set_ylabel("δ<1.25")
    ax.legend(); ax.grid(alpha=0.3)

    fig.suptitle("Frame-Level vs Mask-Level Metrics Comparison (Unscaled)", fontsize=13)
    plt.tight_layout(); _save(fig, "17_frame_vs_mask_metrics.png")

plot_frame_vs_mask_metrics(df_frame, df_mask)

10:27:25  INFO      Saved: 17_frame_vs_mask_metrics.png


In [29]:
# ── Plot 18b: Focal-length distribution (runs only when USE_MODEL_FOCAL=True) ─
def plot_focal_distribution(df: pd.DataFrame) -> None:
    """
    When model-predicted focal is enabled, show:
      Left  — histogram of predicted focal values vs the KITTI default line.
      Right — bar chart of how often each focal_src was used.
    Skipped silently when all rows use kitti_default.
    """
    if "focal_used" not in df.columns or "focal_src" not in df.columns:
        return
    # Use only one clip variant to avoid triple-counting the same frame
    sub = df[df["clip_variant"] == "clip80"].drop_duplicates(subset=["unique_id"])
    if sub.empty:
        return
    if sub["focal_src"].nunique() == 1 and sub["focal_src"].iloc[0] == "kitti_default":
        log.info("All frames used KITTI default focal — skipping focal plot.")
        return

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: histogram of focal values
    ax = axes[0]
    ax.hist(sub["focal_used"], bins=60, color="#5e3c99", alpha=0.80, edgecolor="white")
    ax.axvline(FOCAL_LENGTH_KITTI, color="red", ls="--", lw=2.0,
               label=f"KITTI default = {FOCAL_LENGTH_KITTI:.1f} px")
    ax.axvline(sub["focal_used"].mean(), color="black", ls=":", lw=2.0,
               label=f"Model mean = {sub['focal_used'].mean():.1f} px")
    ax.set_xlabel("Focal Length (pixels)")
    ax.set_ylabel("Frame count")
    ax.set_title("Predicted Focal-Length Distribution")
    ax.legend(fontsize=9)

    # Right: source breakdown
    ax = axes[1]
    src_counts = sub["focal_src"].value_counts()
    src_colors = {"model": "#5e3c99", "kitti_default": "#e66101",
                  "kitti_fallback": "#b2182b"}
    colors = [src_colors.get(s, "grey") for s in src_counts.index]
    bars = ax.bar(src_counts.index, src_counts.values,
                  color=colors, edgecolor="white", width=0.5)
    ax.bar_label(bars, fmt="%d", padding=4, fontsize=11)
    ax.set_ylabel("Number of frames")
    ax.set_title("Focal-Length Source Breakdown")
    ax.grid(axis="y", alpha=0.35)

    fig.suptitle(
        f"Focal-Length Analysis  "
        f"(USE_MODEL_FOCAL = {USE_MODEL_FOCAL})",
        fontsize=13,
    )
    plt.tight_layout()
    _save(fig, "18b_focal_distribution.png")

plot_focal_distribution(df_frame)

10:27:25  INFO      Saved: 18b_focal_distribution.png


In [30]:
# ── Terminal summary ───────────────────────────────────────────────────────────
W = 110
print("═" * W)
print("  DA3METRIC-LARGE — TREND ANALYSIS SUMMARY  (UNSCALED DEPTHS ONLY)")
print(f"  Focal source: {'MODEL-PREDICTED' if USE_MODEL_FOCAL else 'KITTI DEFAULT'} "
      f"| Formula: metric_depth = focal × net_output / 300.")
if "focal_used" in df_frame.columns and USE_MODEL_FOCAL:
    _sub80 = df_frame[df_frame["clip_variant"] == "clip80"]
    _src_counts = _sub80["focal_src"].value_counts().to_dict()
    print(f"  Focal stats (clip80 frames): "
          f"mean={_sub80['focal_used'].mean():.2f} px  "
          f"min={_sub80['focal_used'].min():.2f}  "
          f"max={_sub80['focal_used'].max():.2f}  "
          f"sources={_src_counts}")
print("═" * W)

print("\n[FRAME-LEVEL METRICS]")
hdr = f"{'Clip':>10}  {'#Frames':>8}  {'RMSE(m)':>10}  {'MAE(m)':>10}  "\
      f"{'AbsRel':>10}  {'δ<1.25':>8}  {'Bias(m)':>10}"
print(hdr); print("-" * W)
for clip in CLIP_ORDER:
    sub = df_frame[df_frame["clip_variant"] == clip]
    if sub.empty: continue
    print(f"  {clip:>10}  {len(sub):>8,}  "
          f"{true_rmse(sub, 'rmse'):>10.4f}  {sub['mae'].mean():>10.4f}  "
          f"{sub['abs_rel'].mean():>10.4f}  {sub['delta1'].mean():>8.4f}  "
          f"{sub['mean_error'].mean():>10.4f}")

if not df_mask.empty:
    print("\n[MASK-LEVEL METRICS — CLIP80  (pixel-weighted True RMSE)]")
    df80 = df_mask[df_mask["clip_variant"] == "clip80"]
    cats = sorted(df80["super_cat"].dropna().unique())
    hdr2 = f"  {'Category':>28}  {'#Masks':>8}  {'RMSE(m)':>10}  {'MAE(m)':>10}  "\
           f"{'AbsRel':>10}  {'δ<1.25':>8}  {'Bias(m)':>10}"
    print(hdr2); print("-" * W)
    for cat in cats:
        sub = df80[df80["super_cat"] == cat]
        if sub.empty: continue
        print(f"  {cat:>28}  {len(sub):>8,}  "
              f"{true_rmse(sub, 'rmse'):>10.4f}  {sub['mae'].mean():>10.4f}  "
              f"{sub['abs_rel'].mean():>10.4f}  {sub['delta1'].mean():>8.4f}  "
              f"{sub['mean_error'].mean():>10.4f}")


    # ── All clip variants — overall True RMSE ─────────────────────────────────
    print("\n[MASK-LEVEL OVERALL TRUE RMSE — ALL CLIP VARIANTS]")
    hdr_ov = f"  {'Clip Variant':>14}  {'#Masks':>8}  {'True RMSE (m)':>14}  {'Avg MAE':>10}  {'Avg AbsRel':>12}  {'Avg δ<1.25':>12}"
    print(hdr_ov); print("-" * W)
    for clip in ["clip80", "clip40", "noclip"]:
        sub_clip = df_mask[df_mask["clip_variant"] == clip]
        if sub_clip.empty: continue
        print(f"  {clip:>14}  {len(sub_clip):>8,}  "
              f"{true_rmse(sub_clip, 'rmse'):>14.4f}  {sub_clip['mae'].mean():>10.4f}  "
              f"{sub_clip['abs_rel'].mean():>12.4f}  {sub_clip['delta1'].mean():>12.4f}")

    print("\n[DIFF CATEGORY % — FRAME LEVEL, clip80][LiDAR-Pred_Depth]\n[Positive-> UnderEstimation, Negative-> OverEstimation]")
    df80f = df_frame[df_frame["clip_variant"] == "clip80"]
    pct_cols_present = [c for c in
        ["pct_extreme_pos","pct_too_positive","pct_positive",
         "pct_zero","pct_negative","pct_too_negative","pct_extreme_neg"]
        if c in df80f.columns]
    for c in pct_cols_present:
        lbl = c.replace("pct_", "").replace("_", " ").title()
        print(f"  {lbl:<25}  {df80f[c].mean():>7.2f}%")

print("\n" + "═" * W)

══════════════════════════════════════════════════════════════════════════════════════════════════════════════
  DA3METRIC-LARGE — TREND ANALYSIS SUMMARY  (UNSCALED DEPTHS ONLY)
  Focal source: MODEL-PREDICTED | Formula: metric_depth = focal × net_output / 300.
  Focal stats (clip80 frames): mean=721.56 px  min=721.56  max=721.56  sources={'kitti_fallback': 652}
══════════════════════════════════════════════════════════════════════════════════════════════════════════════

[FRAME-LEVEL METRICS]
      Clip   #Frames     RMSE(m)      MAE(m)      AbsRel    δ<1.25     Bias(m)
--------------------------------------------------------------------------------------------------------------
      clip80       652      2.9382      1.6068      0.1001    0.9345      1.0356
      clip40       652      1.8916      1.2663      0.0977    0.9420      0.8451
      noclip       652      3.0285      1.6181      0.1003    0.9343      1.0320

[MASK-LEVEL METRICS — CLIP80  (pixel-weighted True RMSE)]
         

In [31]:
# ── Plot 18: Diff-category pie charts — frame level per clip ──────────────────
def plot_pie_diff_categories(df: pd.DataFrame) -> None:
    pct_cols = [
        "pct_extreme_pos","pct_too_positive","pct_positive",
        "pct_zero",
        "pct_negative","pct_too_negative","pct_extreme_neg",
    ]
    pct_cols = [c for c in pct_cols if c in df.columns]
    labels   = [DIFF_CAT_LABELS.get(f"n_{c.replace('pct_','')}",
                                     c.replace("pct_","")) for c in pct_cols]
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    for ax, clip in zip(axes, CLIP_ORDER):
        sub  = df[df["clip_variant"] == clip]
        vals = [max(sub[c].mean(), 0) for c in pct_cols]
        ax.pie(vals, labels=labels, colors=DIFF_CAT_COLORS,
               autopct="%1.1f%%", pctdistance=0.82,
               wedgeprops={"edgecolor": "white", "linewidth": 1.0})
        ax.set_title(f"{clip}", fontsize=12)
    fig.suptitle("Difference Category Proportions — Frame Level\nLiDAR - PredDepth (Blue-> UnderEstimation, Red-> OverEstimation)", fontsize=14)
    plt.tight_layout(); _save(fig, "18_pie_diff_categories.png")

plot_pie_diff_categories(df_frame)

10:27:26  INFO      Saved: 18_pie_diff_categories.png


In [32]:
# ── Plot 19: Violin plots — RMSE per category × clip ─────────────────────────
def plot_violin_cat_clip(df: pd.DataFrame) -> None:
    if df.empty or "super_cat" not in df.columns: return
    fig, ax = plt.subplots(figsize=(16, 7))
    sns.violinplot(
        ax=ax, data=df, x="super_cat", y="rmse",
        hue="clip_variant", hue_order=CLIP_ORDER,
        palette=CLIP_PALETTE, split=False,
        inner="quartile", linewidth=0.9,
    )
    ax.set_xlabel("Super Category"); ax.set_ylabel("RMSE (m)")
    ax.set_title("RMSE Distribution — Category × Clip Variant (Unscaled)", fontsize=13)
    ax.tick_params(axis="x", rotation=30)
    ax.legend(title="Clip Variant")
    plt.tight_layout(); _save(fig, "19_violin_cat_clip.png")

if not df_mask.empty:
    plot_violin_cat_clip(df_mask)

10:27:27  INFO      Saved: 19_violin_cat_clip.png


In [33]:
# ── Plot 20: Cumulative RMSE curve — frame level ──────────────────────────────
def plot_cumulative_rmse(df: pd.DataFrame) -> None:
    fig, ax = plt.subplots(figsize=(10, 6))
    for clip in CLIP_ORDER:
        sub  = df[df["clip_variant"] == clip]["rmse"].sort_values()
        cdf  = np.arange(1, len(sub) + 1) / len(sub)
        ax.plot(sub.values, cdf, lw=2,
                color=CLIP_PALETTE[clip], label=clip)
    ax.set_xlabel("RMSE (m)"); ax.set_ylabel("Cumulative Fraction of Frames")
    ax.set_title("Cumulative RMSE Distribution — Frame Level (Unscaled)", fontsize=13)
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); _save(fig, "20_cumulative_rmse.png")

plot_cumulative_rmse(df_frame)

10:27:28  INFO      Saved: 20_cumulative_rmse.png


## 14 · Final Report

In [34]:
# ── Print all output paths ─────────────────────────────────────────────────────
print("\n===  OUTPUT LOCATIONS  ===")
print(f"  Overlay PNGs   : {OVERLAY_DIR}")
print(f"  Frame CSV      : {FRAME_CSV_PATH}")
print(f"  Mask CSV       : {MASK_CSV_PATH}")
print(f"  Analysis plots : {PLOT_DIR}")

plot_files = sorted(Path(PLOT_DIR).glob("*.png"))
print(f"\n  {len(plot_files)} analysis plots saved:")
for p in plot_files:
    print(f"    {p.name}")

print("\n===  DONE  ===")


===  OUTPUT LOCATIONS  ===
  Overlay PNGs   : /STORAGE/Arka/a_MDE_Code/1_Anaysis/Results/DA3_TREND_ANALYSIS_FINAL_modal_focal/overlays
  Frame CSV      : /STORAGE/Arka/a_MDE_Code/1_Anaysis/Results/DA3_TREND_ANALYSIS_FINAL_modal_focal/frame_metrics.csv
  Mask CSV       : /STORAGE/Arka/a_MDE_Code/1_Anaysis/Results/DA3_TREND_ANALYSIS_FINAL_modal_focal/mask_metrics.csv
  Analysis plots : /STORAGE/Arka/a_MDE_Code/1_Anaysis/Results/DA3_TREND_ANALYSIS_FINAL_modal_focal/analysis_plots

  21 analysis plots saved:
    01_summary_table.png
    02_rmse_distribution.png
    03_bias_distribution.png
    04_frame_point_count_bars.png
    05_rmse_vs_depth.png
    06_depth_bin_analysis.png
    07_pred_vs_lidar.png
    08_metric_comparison_bars.png
    09_rmse_by_category.png
    10_rmse_by_cat_clip.png
    11_point_count_by_cat.png
    12_category_boxplots.png
    13_cat_clip_heatmap.png
    14_subcat_rmse.png
    15_subcat_diff_heatmap.png
    16_mask_rmse_vs_depth_by_cat.png
    17_frame_vs_mask_met